In [0]:
from pyspark.sql import functions as F

segment_by_brand = spark.table("prod_latam_catalog.crm_reporting_bkp.fact_segment_by_brand_bkp_20250307")
# segment_by_brand = spark.table("crm_reporting.fact_segment_by_brand")
segment_by_brand.createOrReplaceTempView("segment_by_brand_vw")

In [0]:
%sql
select distinct lifetime_new_buyer_activity_segment_key--lifetime_new_consumer_activity_segment_key
from segment_by_brand_vw

lifetime_new_buyer_activity_segment_key
DDM_LNBAS_12PB
DDM_LNBAS_46B
DDM_LNBAS_1B
DDM_LNBAS_NA
DDM_LNBAS_3B
DDM_LNBAS_1012B
DDM_LNBAS_79B
DDM_LNBAS_2B


In [0]:
# %sql
# select distinct
#   interaction_sub_type
# -- from segment_by_brand_vw
# from prod_latam_catalog.crm_reporting.fact_customer_interactions
# -- where brand_customer_id = '71879afca413036f66ae69455616cc35'

In [0]:
# ## DETALLE DE DUPLICADOS
# tmp = spark.sql("""
# with tmps as (
#   select distinct
#     brand_customer_id,
#     lifetime_buyer_activity_segment_key
#   from segment_by_brand_vw
# )

# select brand_customer_id, count(brand_customer_id) as counts
# from tmps
# group by brand_customer_id
# having counts > 1
# order by counts desc
# --limit 1000
# """)

# display(tmp)
# tmp.createOrReplaceTempView("tmp_vw")

In [0]:
### VARIABLES
# brand_country = 'ARG'
# brand_code = 'LOP'
snapshot_prev = '20241231'
snapshot_next = '20250120'

# QUERY OFICIAL
query = f"""
with snap_next as (
  select distinct
    brand_country,
    brand_code,
    brand_customer_id,
    lifetime_new_consumer_activity_segment_key
  from segment_by_brand_vw
  where snapshot_date_key = '{snapshot_next}'
),

snap_prev as (
  select distinct
    brand_country,
    brand_code,
    brand_customer_id,
    lifetime_new_consumer_activity_segment_key
  from segment_by_brand_vw
  where snapshot_date_key = '{snapshot_prev}' 
),

-- ids que no estan en el siguiente snapshot y se consiguen por la resta, en teoria, los nuevos
ids_x_subtr as (
  select
    a.brand_country,
    a.brand_code,
    a.brand_customer_id as brand_customer_id_jun,
    a.lifetime_new_consumer_activity_segment_key as segment,
    b.brand_customer_id as brand_customer_id_may
  from snap_next a
  left join snap_prev b
    on a.brand_customer_id = b.brand_customer_id
    and a.brand_code = b.brand_code 
    and a.brand_country = b.brand_country
  -- in this line I just take into account new clients --
  where b.brand_customer_id is null
),

flagged_snap1 as (
  select distinct
    brand_country,
    brand_code,
    brand_customer_id,
    lifetime_new_consumer_activity_segment_key as segment
  from segment_by_brand_vw
  where snapshot_date_key = '{snapshot_next}'
    and lifetime_new_consumer_activity_segment_key = 'DDM_LNCAS_1C' 
),

-- cruce de los nuevos por resta vs los que tienen flag de nuevos
ids_faltantes_vw as (
  select
    case when a.brand_country is null then b.brand_country else a.brand_country end as brand_country,
    case when a.brand_code is null then b.brand_code else a.brand_code end as brand_code,
    a.brand_customer_id as brand_customer_id_flag,
    b.brand_customer_id_jun as brand_customer_id_subtr,
    case when a.segment is null then b.segment else a.segment end as segment
  from flagged_snap1 a
  full join ids_x_subtr b
    on a.brand_customer_id = b.brand_customer_id_jun
  where b.brand_customer_id_may is null
  order by brand_customer_id_flag
),

ids_flageados_sin_estar_en_resta as (
  select distinct
    brand_country,
    brand_code,
    brand_customer_id_flag as brand_customer_id,
    segment,
    'flagged not in subtr' as status
  from ids_faltantes_vw
  where brand_customer_id_subtr is null
),

ids_no_flageados as (
  select distinct
    brand_country,
    brand_code,
    brand_customer_id_subtr as brand_customer_id,
    segment,
    'not flagged' as status
  from ids_faltantes_vw
  where brand_customer_id_flag is null 
),

ids_flageados as (
  select distinct
    brand_country,
    brand_code,
    brand_customer_id_flag as brand_customer_id,
    segment,
    'correct' as status
  from ids_faltantes_vw
  where brand_customer_id_flag is not null 
    and brand_customer_id_subtr is not null
),

unions as (
  select * from ids_flageados_sin_estar_en_resta
  union 
  select * from ids_no_flageados
  union 
  select * from ids_flageados
),

customer_bridge as (
  select distinct
    brand_country,
    brand_code,
    brand_customer_id,
    source_name,
    source_customer_id
  from prod_latam_catalog.crm_reporting.dim_customer_bridge
),

-- tomamos el source_customer_id de la bridge
cruce_customer_bridge as (
  select 
    a.*,
    b.source_name,
    b.source_customer_id 
  from unions a
  left join customer_bridge b
    on a.brand_customer_id = b.brand_customer_id
    and a.brand_code = b.brand_code 
    and a.brand_country = b.brand_country
),

interactions as (
  select distinct
    brand_country,
    brand_code,
    --source_name,
    source_customer_id,
    interaction_sub_type,
    to_date(interaction_dt) as interaction_dt
  from prod_latam_catalog.crm_reporting.fact_customer_interactions
  where upper(interaction_sub_type) = 'PROFILE CREATED'
),

ids_faltantes_detalle_vw as (
  select 
    a.*,
    b.interaction_sub_type,
    b.interaction_dt
  from cruce_customer_bridge a
  left join interactions b
    on a.source_customer_id = b.source_customer_id
    --and a.source_name = b.source_name
    and a.brand_code = b.brand_code 
    and a.brand_country = b.brand_country
  where b.interaction_sub_type is not null
  order by brand_customer_id,interaction_dt
),

ids_faltantes_detalle as (
  select distinct
    brand_country,
    brand_code,
    source_name,
    brand_customer_id,
    segment,
    status,
    FIRST_VALUE(source_customer_id) 
      --OVER (PARTITION BY brand_country,brand_code,source_name,brand_customer_id ORDER BY interaction_dt asc) AS source_customer_id,
      OVER (PARTITION BY brand_country,brand_code,brand_customer_id ORDER BY interaction_dt asc) AS source_customer_id,
    interaction_sub_type,
    FIRST_VALUE(interaction_dt) 
      --OVER (PARTITION BY brand_country,brand_code,source_name,brand_customer_id ORDER BY interaction_dt asc) AS interaction_dt
      OVER (PARTITION BY brand_country,brand_code,brand_customer_id ORDER BY interaction_dt asc) AS interaction_dt
  from ids_faltantes_detalle_vw
),

dim_customer as (
  select distinct
    brand_country,
    brand_code,
    source_name,
    source_customer_id,
    FIRST_VALUE(created_dt) 
      --OVER (PARTITION BY brand_country,brand_code,source_name,source_customer_id ORDER BY created_dt asc) AS created_dt_dim_customer
      OVER (PARTITION BY brand_country,brand_code,source_customer_id ORDER BY created_dt asc) AS created_dt_dim_customer
  --from prod_latam_catalog.crm_reporting.dim_customer
  from prod_latam_catalog.crm_reporting.dim_customer_hist
),

brand_profile as (
  select distinct
    brand_country,
    brand_code,
    brand_customer_id,
    FIRST_VALUE(created_dt) 
      OVER (PARTITION BY brand_country,brand_code,brand_customer_id ORDER BY created_dt asc) AS created_dt_brand_profile
  from prod_latam_catalog.crm_reporting.dim_customer_brand_profile
)

select 
  '{snapshot_next}' as snapshot,
  a.*,
  to_date(b.created_dt_dim_customer) as created_dt_dim_customer,
  to_date(c.created_dt_brand_profile) as created_dt_brand_profile
from ids_faltantes_detalle a
left join dim_customer b
  on a.source_customer_id = b.source_customer_id
  and a.source_name = b.source_name
  and a.brand_code = b.brand_code 
  and a.brand_country = b.brand_country
left join brand_profile c
  on a.brand_customer_id = c.brand_customer_id
  and a.brand_code = c.brand_code 
  and a.brand_country = c.brand_country
order by status

"""


ids_faltantes = spark.sql(query)
ids_faltantes.createOrReplaceTempView("ids_faltantes_vw")
# display(ids_faltantes)
# print(ids_faltantes.count())


In [0]:
# Para crear una tabla permanente en el catalogo
ids_faltantes.write.mode('overwrite').option("mergeSchema", "true").format("delta").saveAsTable("prod_latam_catalog.crm_analytics.diagnosis_new_clients")

In [0]:
snapshot_prev = '20241231'
snapshot_next = '20250131'
month_year = '12_2024'


query = f"""
with subtraction as (
  select
    brand_country,
    brand_code,
    count(distinct case when snapshot_date_key = '{snapshot_next}' then brand_customer_id end) 
      - count(distinct case when snapshot_date_key = '{snapshot_prev}' then brand_customer_id end)  as new_contacts_subtr
  from segment_by_brand_vw
  group by all
),

kpis_1 as (
  select *,
  concat(month(interaction_dt),'_',year(interaction_dt)) as my_interaction,
  concat(month(created_dt_dim_customer),'_',year(created_dt_dim_customer)) as my_dim_customer,
  concat(month(created_dt_brand_profile),'_',year(created_dt_brand_profile)) as my_brand_profile
from ids_faltantes_vw
),

kpis as (
  select
    snapshot,
    brand_country,
    brand_code,
    count(distinct brand_customer_id) as total_contacts,
    count(distinct case when my_interaction = '{month_year}' then brand_customer_id end) as my_interaction,
    count(distinct case when my_dim_customer = '{month_year}' then brand_customer_id end) as my_dim_customer,
    count(distinct case when my_brand_profile = '{month_year}' then brand_customer_id end) as my_brand_profile,
    count(distinct case when status = 'correct' then brand_customer_id end) as correct_contacts,
    count(distinct case when status = 'flagged not in subtr' then brand_customer_id end) as half_correct_contacts,
    count(distinct case when status = 'not flagged' then brand_customer_id end) as incorrect_contacts,
    count(distinct case when interaction_sub_type is null then brand_customer_id end) as not_in_bridge_contacts
  from kpis_1
  group by all
  order by brand_country,brand_code
)

select
  a.brand_country,
  a.brand_code,
  a.total_contacts,
  b.new_contacts_subtr,
  --round(b.new_contacts_subtr/b.new_contacts_subtr *100,1) as new_contacts_subtr_r,
  --a.my_interaction,
  round(a.my_interaction/b.new_contacts_subtr *100,1) as my_interaction_r,
  --a.my_dim_customer,
  round(a.my_dim_customer/b.new_contacts_subtr *100,1) as my_dim_customer_r,
  --a.my_brand_profile,
  round(a.my_brand_profile/b.new_contacts_subtr *100,1) as my_brand_profile_r
from kpis a
left join subtraction b
  on a.brand_country = b.brand_country
  and a.brand_code = b.brand_code
order by brand_country,brand_code
"""
kpis = spark.sql(query)
display(kpis)

### VALIDATION

In [0]:
snapshot_prev = '20241231'
snapshot_next = '20250131'

query = f"""
with analysis as (
  select
    brand_country,
    brand_code,
    count(distinct case when snapshot_date_key = '{snapshot_next}' then brand_customer_id end) 
      - count(distinct case when snapshot_date_key = '{snapshot_prev}' then brand_customer_id end)  as new_contacts_substr,
    0 as new_contacts_flag
  from crm_reporting.fact_segment_by_brand
  group by all

  union all

  select
    brand_country,
    brand_code,
    0 as new_contacts_substr,
    count(distinct case when lifetime_new_consumer_activity_segment_key = 'DDM_LNCAS_1C' then brand_customer_id end) as new_contacts_flag
  from crm_reporting.fact_segment_by_brand
  where snapshot_date_key in ('{snapshot_next}')
  group by all


)
select
  brand_country,
  brand_code,
  sum(new_contacts_substr) as new_contacts_substr,
  sum(new_contacts_flag) as new_contacts_flag,
  round(sum(new_contacts_flag) / sum(new_contacts_substr),2) as new_contacts_diff_rate -- the closest to 1, the better
from analysis
group by all
having sum(new_contacts_substr) > 0 
  and sum(new_contacts_substr) > 0
order by brand_country,brand_code
"""
result_df = spark.sql(query)
display(result_df)



brand_country,brand_code,new_contacts_substr,new_contacts_flag,new_contacts_diff_rate
ARG,CER,3819,3819,1.0
ARG,KER,132,12,0.09
ARG,KIE,305,308,1.01
ARG,LAN,559,575,1.03
ARG,LOP,24,23,0.96
ARG,LRP,3571,3551,0.99
ARG,VIC,2277,2269,1.0
BRA,DMC,58539,59484,1.02
BRA,KER,5920,5162,0.87
BRA,LAN,1784,1868,1.05


In [0]:
# brand_country = 'ARG'
# brand_code = 'KER'
# brand_customer_id = '0182563cdf7b329be1785fa9f947f13a' -- not in bridge

# brand_country = 'BRA'
# brand_code = 'LRP'
# brand_customer_id = '45bb11274afdf01d30391659b92dc5c4' 

brand_country = 'MEX'
brand_code = 'LAN'
brand_customer_id = '76166eabe7d1e7d2b76abfc82939eb0a' # flagged but not in interactions

#### ids_faltantes_vw
query = f"""
select *
from ids_faltantes_vw
WHERE brand_customer_id = '{brand_customer_id}'
"""
result_df = spark.sql(query)
display(result_df)

# variable source_customer_id
source_customer_id = result_df.collect()[0]['source_customer_id']



#### dim_customer_brand_profile
query = f"""
select 'brand_profile' as source,brand_country,brand_code,brand_customer_id, date(created_dt) as created_dt
from prod_latam_catalog.crm_reporting.dim_customer_brand_profile
WHERE brand_customer_id = '{brand_customer_id}'
  AND brand_country = '{brand_country}'  
  AND brand_code = '{brand_code}'
"""
result_df = spark.sql(query)
display(result_df)

#### dim_customer_bridge
query = f"""
select 'bridge' as source,brand_code,brand_country,source_name,source_customer_id,brand_customer_id
from prod_latam_catalog.crm_reporting.dim_customer_bridge
where source_customer_id = '{source_customer_id}' 
  and brand_country = '{brand_country}'  
  and brand_code = '{brand_code}'
"""
result_df = spark.sql(query)
display(result_df)

#### dim_customer
query = f"""
select 'dim_customer' as source,brand_country,brand_code,source_name,source_customer_id,date(created_dt) as created_dt
from prod_latam_catalog.crm_reporting.dim_customer
where source_customer_id = '{source_customer_id}' 
  and brand_country = '{brand_country}'  
  and brand_code = '{brand_code}'
"""
result_df = spark.sql(query)
display(result_df)


#### fact_customer_interactions
query = f"""
select 'interactions' as source,source_customer_id,brand_country,brand_code,source_name,interaction_sub_type,date(interaction_dt) as interaction_dt
from prod_latam_catalog.crm_reporting.fact_customer_interactions
where source_customer_id = '{source_customer_id}' 
  and brand_country = '{brand_country}'  
  and brand_code = '{brand_code}'
  and UPPER(interaction_sub_type) = 'PROFILE CREATED'
"""
result_df = spark.sql(query)
display(result_df)



In [0]:
%sql
Select *
from ids_faltantes_vw  -- prod_latam_catalog.crm_reporting.fact_segment_by_brand
WHERE brand_customer_id = '3cfcf04d9e58cc4fb8b3bad0636fca83'
-- WHERE source_customer_id = 'f1504164bf616a8b99356fcd2056ce99'
order by interaction_dt

snapshot,brand_country,brand_code,source_name,brand_customer_id,segment,status,source_customer_id,interaction_sub_type,interaction_dt,created_dt_dim_customer,created_dt_brand_profile
20240630,CHI,LAN,DEMANDWARE,3cfcf04d9e58cc4fb8b3bad0636fca83,DDM_LNCAS_1C,correct,589496032c6b74b0fe9bf374dd897338,Profile Created,2024-06-19,null,null


In [0]:
%sql
select distinct snapshot_date_key,brand_country,brand_code,brand_customer_id,lifetime_new_consumer_activity_segment_key
-- from crm_reporting.fact_segment_by_brand
from prod_latam_catalog.crm_reporting_bkp.fact_segment_by_brand_bkp_20241205
where brand_customer_id = '0b84f682c3a17663c07a2d00c9446f96' 
  --and snapshot_date_key = '20240630'
order by snapshot_date_key

snapshot_date_key,brand_country,brand_code,brand_customer_id,lifetime_new_consumer_activity_segment_key
20240630,ARG,LOP,0b84f682c3a17663c07a2d00c9446f96,DDM_LNCAS_12PC
20240731,ARG,LOP,0b84f682c3a17663c07a2d00c9446f96,DDM_LNCAS_12PC
20240831,ARG,LOP,0b84f682c3a17663c07a2d00c9446f96,DDM_LNCAS_12PC
20240930,ARG,LOP,0b84f682c3a17663c07a2d00c9446f96,DDM_LNCAS_12PC
20241031,ARG,LOP,0b84f682c3a17663c07a2d00c9446f96,DDM_LNCAS_12PC
20241110,ARG,LOP,0b84f682c3a17663c07a2d00c9446f96,DDM_LNCAS_12PC
20241111,ARG,LOP,0b84f682c3a17663c07a2d00c9446f96,DDM_LNCAS_12PC
20241112,ARG,LOP,0b84f682c3a17663c07a2d00c9446f96,DDM_LNCAS_12PC
20241113,ARG,LOP,0b84f682c3a17663c07a2d00c9446f96,DDM_LNCAS_12PC
20241114,ARG,LOP,0b84f682c3a17663c07a2d00c9446f96,DDM_LNCAS_12PC


In [0]:
%sql
select --* 
brand_code,brand_country,brand_customer_id,created_dt,is_deleted,last_modified_dt
from prod_latam_catalog.crm_reporting.dim_customer_brand_profile
WHERE brand_customer_id = '17274d315f3e64e6f61c9619f9f659fe' 

brand_code,brand_country,brand_customer_id,created_dt,is_deleted,last_modified_dt
DMC,BRA,17274d315f3e64e6f61c9619f9f659fe,2025-01-14T00:15:27Z,N,2025-01-22T00:45:59Z


In [0]:
%sql
select source_customer_id,brand_country,brand_code,interaction_sub_type,date(interaction_dt) as interaction_dt,source_name
from prod_latam_catalog.crm_reporting.fact_customer_interactions
-- WHERE source_customer_id in ('6eddb4ab90a32d1bd5473ea62bda819a','589496032c6b74b0fe9bf374dd897338') 
where source_customer_id in ('0d229b839b8ebca9f01db81a91e06c27')
  and interaction_sub_type = 'Profile Created' 
  -- and brand_country = 'CHI'  
  -- and brand_code = 'KER'

source_customer_id,brand_country,brand_code,interaction_sub_type,interaction_dt,source_name
0d229b839b8ebca9f01db81a91e06c27,BRA,DMC,Profile Created,2025-01-14,DERMACLUB


In [0]:
%sql
select --*
  brand_code,brand_country,source_customer_id,brand_customer_id
-- from prod_latam_catalog.crm_reporting.dim_customer_bridge
from prod_latam_catalog.crm_reporting.etl_stg_customer_bridge
where brand_customer_id = '0b84f682c3a17663c07a2d00c9446f96' 
-- where source_customer_id = 'cfd3f49fa20873d91c1820da87f938f6'

brand_code,brand_country,source_customer_id,brand_customer_id
LOP,ARG,dd141d59cc2707811d85dc6478097604,0b84f682c3a17663c07a2d00c9446f96


In [0]:
%sql
select brand_country,brand_code,source_name,source_customer_id,brand_customer_id,brand_mdm_id
from prod_latam_catalog.crm_reporting.dim_customer_bridge
-- from prod_latam_catalog.crm_reporting.etl_stg_customer_bridge
-- where brand_customer_id = '3cfcf04d9e58cc4fb8b3bad0636fca83' 
where source_customer_id = '0001cf78833f0d927843e0ef7e8fdc21'
order by brand_country,brand_code,source_name

brand_country,brand_code,source_name,source_customer_id,brand_customer_id,brand_mdm_id
CHI,KER,DEMANDWARE,0001cf78833f0d927843e0ef7e8fdc21,b1c69381c04512ce3b495179656282c8,ff0be336fafc912534aadcdad9452b69
CHI,KER,SFMC_FACEBOOK_LEADAD,0001cf78833f0d927843e0ef7e8fdc21,b1c69381c04512ce3b495179656282c8,ff0be336fafc912534aadcdad9452b69
CHI,KIE,SFMC_FACEBOOK_LEADAD,0001cf78833f0d927843e0ef7e8fdc21,b9c5f25410a4baf4694331b43ea5b4d2,e8c8b7ee0992ec022d4a9cd4e90c8c44
CHI,LRP,SFMC_FACEBOOK_LEADAD,0001cf78833f0d927843e0ef7e8fdc21,c85bd4ec47e4139ec709a188e29176f4,61e9ceab7a1b233c31455a25ee3380d8
CHI,VIC,SFMC_FACEBOOK_LEADAD,0001cf78833f0d927843e0ef7e8fdc21,4018a8e06cd9590393f40a6b08399a06,ad6421be685fc6be0ca0ee8da9c26baf


In [0]:
%sql
select --*
  brand_country,brand_code,source_name,source_customer_id,date(created_dt) as created_dt,date(last_modified_dt) as last_modified_dt
from prod_latam_catalog.crm_reporting.dim_customer
where source_customer_id = '0001cf78833f0d927843e0ef7e8fdc21' 
  and brand_country = 'CHI'  
  and brand_code = 'KER'
order by created_dt

brand_country,brand_code,source_name,source_customer_id,created_dt,last_modified_dt
CHI,KER,DEMANDWARE,0001cf78833f0d927843e0ef7e8fdc21,2023-12-11,2023-12-11


In [0]:
%sql
select 
  brand_country,brand_code,source_name,source_customer_id,customer_hist_id,date(created_dt) as created_dt,date(last_modified_dt) as last_modified_dt,sys_last_modified_dt 
from prod_latam_catalog.crm_reporting.dim_customer_hist
where source_customer_id in ('0001cf78833f0d927843e0ef7e8fdc21') 
  and brand_country = 'CHI'  
  and brand_code = 'KER'
-- order by created_dt
order by customer_hist_id desc,last_modified_dt desc,sys_last_modified_dt desc

brand_country,brand_code,source_name,source_customer_id,customer_hist_id,created_dt,last_modified_dt,sys_last_modified_dt
CHI,KER,DEMANDWARE,0001cf78833f0d927843e0ef7e8fdc21,fee85ddad55bb3cd8dbfed7e6dc699d9_20240215073520.408,2023-12-11,2023-12-11,2024-02-15T09:08:34Z
CHI,KER,DEMANDWARE,0001cf78833f0d927843e0ef7e8fdc21,fee85ddad55bb3cd8dbfed7e6dc699d9_20231211162324.027,2023-12-11,2023-12-11,2023-12-11T18:06:51Z
CHI,KER,SFMC_FACEBOOK_LEADAD,0001cf78833f0d927843e0ef7e8fdc21,b389e65425c23e7dc68414e9e6b35826_20240711120826.765,2024-06-30,2024-06-30,2024-07-11T12:08:26.765Z
CHI,KER,SFMC_FACEBOOK_LEADAD,0001cf78833f0d927843e0ef7e8fdc21,b389e65425c23e7dc68414e9e6b35826_20240626120812.306,2024-06-24,2024-06-24,2024-06-26T12:08:12.306Z
CHI,KER,SFMC_FACEBOOK_LEADAD,0001cf78833f0d927843e0ef7e8fdc21,b389e65425c23e7dc68414e9e6b35826_20240111131006.680,2023-12-01,2023-12-01,2024-01-11T13:10:06.68Z
CHI,KER,SFMC_FACEBOOK_LEADAD,0001cf78833f0d927843e0ef7e8fdc21,b389e65425c23e7dc68414e9e6b35826_20240111131006.680,2023-12-01,2023-12-01,2024-01-11T13:10:06.68Z
CHI,KER,SFMC_FACEBOOK_LEADAD,0001cf78833f0d927843e0ef7e8fdc21,b389e65425c23e7dc68414e9e6b35826_20231209130744.256,2023-12-07,2023-12-07,2023-12-09T13:07:44.256Z
CHI,KER,SFMC_FACEBOOK_LEADAD,0001cf78833f0d927843e0ef7e8fdc21,b389e65425c23e7dc68414e9e6b35826_20231209130744.256,2023-12-07,2023-12-07,2023-12-09T13:07:44.256Z


In [0]:
# %sql
# describe prod_latam_catalog.crm_reporting.dim_customer_brand_profile

In [0]:
# %sql
# describe prod_latam_catalog.crm_reporting.vw_fact_segment_by_brand

In [0]:
%sql
select distinct
  brand_customer_id,
  snapshot_date_key,
  lifetime_new_consumer_activity_segment_key
-- from segment_by_brand_vw
from prod_latam_catalog.crm_reporting.vw_fact_segment_by_brand
where brand_customer_id = '050c9d97b4a90cadfc85900657be59de'

brand_customer_id,snapshot_date_key,lifetime_new_consumer_activity_segment_key
050c9d97b4a90cadfc85900657be59de,20240831,DDM_LNCAS_12PC
050c9d97b4a90cadfc85900657be59de,20241231,DDM_LNCAS_12PC
050c9d97b4a90cadfc85900657be59de,20231231,DDM_LNCAS_79C
050c9d97b4a90cadfc85900657be59de,20250119,DDM_LNCAS_12PC
050c9d97b4a90cadfc85900657be59de,20250116,DDM_LNCAS_12PC
050c9d97b4a90cadfc85900657be59de,20250121,DDM_LNCAS_12PC
050c9d97b4a90cadfc85900657be59de,20230531,DDM_LNCAS_2C
050c9d97b4a90cadfc85900657be59de,20240731,DDM_LNCAS_12PC
050c9d97b4a90cadfc85900657be59de,20250114,DDM_LNCAS_12PC
050c9d97b4a90cadfc85900657be59de,20240331,DDM_LNCAS_1012C


In [0]:
# %sql
# select
#   brand_country,
#   brand_code,
#   status,
#   count(distinct brand_customer_id) as counts
# from ids_faltantes_vw
# group by all
# order by brand_country,
#   brand_code,
#   status

In [0]:
%sql
-- describe prod_latam_catalog.crm_reporting.dim_customer_bridge
describe prod_latam_catalog.crm_reporting.fact_customer_interactions

col_name,data_type,comment
brand_code,string,null
brand_country,string,null
interaction_year,int,null
interaction_month,string,null
source_name,string,null
source_customer_id,string,null
interaction_type,string,null
interaction_sub_type,string,null
interaction_dt,timestamp,null
interaction_day,int,null


In [0]:
%sql
select * from
(select distinct
a.brand_country,
a.brand_code,
c.is_deleted,
concat(year(a.created_dt),'_',month(a.created_dt)) as year_month,
a.source_customer_id as source_customer_id_dc,
b.source_customer_id as source_customer_id_dcb
from prod_latam_catalog.crm_reporting.dim_customer a
inner join prod_latam_catalog.crm_reporting.dim_customer_extn c
on a.source_customer_id = c.source_customer_id
and a.brand_code = c.brand_code
and a.brand_country = c.brand_country
left join prod_latam_catalog.crm_reporting.dim_customer_bridge b
on a.source_customer_id = b.source_customer_id
and a.brand_code = b.brand_code
and a.brand_country = b.brand_country
where upper(a.source_name) IN ('DEMANDWARE','JEBBIT')
AND (lower(a.registration_source) LIKE ('%quiz%') OR lower(a.registration_source) LIKE ('%diagnos%'))
AND to_date(a.created_dt) >= '2023-01-01')
where source_customer_id_dcb is null
and is_deleted is null;

brand_country,brand_code,is_deleted,year_month,source_customer_id_dc,source_customer_id_dcb
BRA,LRP,null,2023_6,06a5b53061d27c3b8aa3138e7f601313,null
BRA,LRP,null,2023_6,19b2eba4b9317ffa52b53b0081d020ed,null
BRA,LRP,null,2023_6,1f51ed2e306b5c72dd6ca9681fec5045,null
BRA,LRP,null,2023_6,23ed75177a7417ba79d4b493ff1705aa,null
BRA,LRP,null,2023_6,2f0434895861e051efac409d30091324,null
BRA,LRP,null,2023_6,3c1f7d79c341e78ff2207728371f9285,null
BRA,LRP,null,2023_6,460ecca0d89dd9c93fa1dc6beb559f10,null
BRA,LRP,null,2023_6,4aa5d3530a2ccfedea2265a63662f2eb,null
BRA,LRP,null,2023_6,54d75402e483b9e18ebd478e2f69bbfb,null
BRA,LRP,null,2023_6,5815b22ed05af9a61946e070ce96778e,null


In [0]:
# ## DETALLE DE DUPLICADOS
# tmp = spark.sql("""
# select brand_customer_id , count(distinct source_customer_id) as counts
# from prod_latam_catalog.crm_reporting.dim_customer_bridge
# group by brand_customer_id 
# having counts > 1
# order by counts desc
# --limit 1000
# """)

# display(tmp)
# tmp.createOrReplaceTempView("tmp_vw")

In [0]:
%sql
select brand_country,brand_code,source_name,source_customer_id,brand_customer_id,brand_mdm_id
from prod_latam_catalog.crm_reporting.dim_customer_bridge
-- from prod_latam_catalog.crm_reporting.etl_stg_customer_bridge
where brand_customer_id = '96d9c9a03e4dd65c253902188d9f5135'
-- where source_customer_id = '0001cf78833f0d927843e0ef7e8fdc21'
order by brand_country,brand_code,source_name

brand_country,brand_code,source_name,source_customer_id,brand_customer_id,brand_mdm_id
BRA,LAN,DEMANDWARE,8243a49d274ec28d6916943d92c7aa8f,96d9c9a03e4dd65c253902188d9f5135,10369cf52af4a9da3ae3f72ea917c800
BRA,LAN,DEMANDWARE,10f1cd087e4da05e73c52011ef9004ed,96d9c9a03e4dd65c253902188d9f5135,10369cf52af4a9da3ae3f72ea917c800
BRA,LAN,DEMANDWARE,98c855a6ca9f13c202561392fe2a1982,96d9c9a03e4dd65c253902188d9f5135,10369cf52af4a9da3ae3f72ea917c800
BRA,LAN,SFMC_ECOMMERCE_VTEX,42834b4b79994e69064ffbc6c536d1d8,96d9c9a03e4dd65c253902188d9f5135,10369cf52af4a9da3ae3f72ea917c800
BRA,LAN,SFMC_ECOMMERCE_VTEX,db316d65a24c8247ad963ff131b206be,96d9c9a03e4dd65c253902188d9f5135,10369cf52af4a9da3ae3f72ea917c800
BRA,LAN,VTEX,42834b4b79994e69064ffbc6c536d1d8,96d9c9a03e4dd65c253902188d9f5135,10369cf52af4a9da3ae3f72ea917c800
BRA,LAN,VTEX,db316d65a24c8247ad963ff131b206be,96d9c9a03e4dd65c253902188d9f5135,10369cf52af4a9da3ae3f72ea917c800
BRA,LAN,VTEX,42f3ebde32bba259091c8901790ba382,96d9c9a03e4dd65c253902188d9f5135,10369cf52af4a9da3ae3f72ea917c800


In [0]:
%sql
select source_customer_id,brand_country,brand_code,interaction_sub_type,date(interaction_dt) as interaction_dt,source_name
from prod_latam_catalog.crm_reporting.fact_customer_interactions
-- WHERE source_customer_id in ('6eddb4ab90a32d1bd5473ea62bda819a','589496032c6b74b0fe9bf374dd897338') 
where source_customer_id in ('8243a49d274ec28d6916943d92c7aa8f','8243a49d274ec28d6916943d92c7aa8f','10f1cd087e4da05e73c52011ef9004ed',
'98c855a6ca9f13c202561392fe2a1982','42834b4b79994e69064ffbc6c536d1d8','db316d65a24c8247ad963ff131b206be','42f3ebde32bba259091c8901790ba382')
  and interaction_sub_type = 'Profile Created' 
  and brand_country = 'BRA'  
  and brand_code = 'LAN'

source_customer_id,brand_country,brand_code,interaction_sub_type,interaction_dt,source_name
10f1cd087e4da05e73c52011ef9004ed,BRA,LAN,Profile Created,2023-03-01,DEMANDWARE
8243a49d274ec28d6916943d92c7aa8f,BRA,LAN,Profile Created,2023-03-01,DEMANDWARE
98c855a6ca9f13c202561392fe2a1982,BRA,LAN,Profile Created,2023-03-01,DEMANDWARE
db316d65a24c8247ad963ff131b206be,BRA,LAN,Profile Created,2022-01-22,SFMC_ECOMMERCE_VTEX
db316d65a24c8247ad963ff131b206be,BRA,LAN,Profile Created,2022-05-19,VTEX
42834b4b79994e69064ffbc6c536d1d8,BRA,LAN,Profile Created,2021-10-21,SFMC_ECOMMERCE_VTEX
42834b4b79994e69064ffbc6c536d1d8,BRA,LAN,Profile Created,2022-04-28,VTEX
42f3ebde32bba259091c8901790ba382,BRA,LAN,Profile Created,2022-11-24,VTEX


In [0]:
%sql
select --*
  brand_country,brand_code,source_name,source_customer_id,date(created_dt) as created_dt,date(last_modified_dt) as last_modified_dt
from prod_latam_catalog.crm_reporting.dim_customer_hist
where source_customer_id in ('8243a49d274ec28d6916943d92c7aa8f','8243a49d274ec28d6916943d92c7aa8f','10f1cd087e4da05e73c52011ef9004ed',
'98c855a6ca9f13c202561392fe2a1982','42834b4b79994e69064ffbc6c536d1d8','db316d65a24c8247ad963ff131b206be','42f3ebde32bba259091c8901790ba382')
  and brand_country = 'BRA'  
  and brand_code = 'LAN'
order by created_dt

brand_country,brand_code,source_name,source_customer_id,created_dt,last_modified_dt
BRA,LAN,SFMC_ECOMMERCE_VTEX,42834b4b79994e69064ffbc6c536d1d8,2021-10-21,2021-10-21
BRA,LAN,DEMANDWARE,98c855a6ca9f13c202561392fe2a1982,2021-10-21,2023-03-01
BRA,LAN,VTEX,42834b4b79994e69064ffbc6c536d1d8,2021-10-22,2022-04-28
BRA,LAN,DEMANDWARE,8243a49d274ec28d6916943d92c7aa8f,2022-01-22,2023-03-01
BRA,LAN,SFMC_ECOMMERCE_VTEX,db316d65a24c8247ad963ff131b206be,2022-01-22,2022-01-22
BRA,LAN,VTEX,db316d65a24c8247ad963ff131b206be,2022-01-22,2022-11-22
BRA,LAN,VTEX,db316d65a24c8247ad963ff131b206be,2022-01-22,2022-05-19
BRA,LAN,VTEX,db316d65a24c8247ad963ff131b206be,2022-01-22,2022-11-24
BRA,LAN,DEMANDWARE,10f1cd087e4da05e73c52011ef9004ed,2022-11-24,2023-03-01
BRA,LAN,VTEX,42f3ebde32bba259091c8901790ba382,2022-11-24,2022-11-24


In [0]:
query = f"""
with dim_customer_hist as (
  select distinct
    brand_country,
    brand_code,
    source_customer_id,
    FIRST_VALUE(date(created_dt)) 
      OVER (PARTITION BY brand_country,brand_code,source_customer_id ORDER BY created_dt asc) AS created_dt
  from prod_latam_catalog.crm_reporting.dim_customer_hist
),

interactions as (
  select distinct
    brand_country,
    brand_code,
    source_customer_id,
    FIRST_VALUE(date(interaction_dt)) 
      OVER (PARTITION BY brand_country,brand_code,source_customer_id ORDER BY interaction_dt asc) AS interaction_dt
  from prod_latam_catalog.crm_reporting.fact_customer_interactions
  where interaction_sub_type = 'Profile Created' 
)--,

--cruce as (
select
  a.*,
  b.interaction_dt,
  case when a.created_dt = b.interaction_dt then 1 else 0 end as status
from dim_customer_hist a
left join interactions b
  on a.source_customer_id = b.source_customer_id
  and a.brand_code = b.brand_code 
  and a.brand_country = b.brand_country
where year(b.interaction_dt) == 2024 and month(b.interaction_dt) == 6
--)
"""

tb1 = spark.sql(query)
# display(tb1)
tb1.createOrReplaceTempView("tb1_vw")


In [0]:
%sql
select *
from tb1_vw
where status = 0

brand_country,brand_code,source_customer_id,created_dt,interaction_dt,status
BRA,DMC,b6b241eb7d69a5f874a9a93f52be373c,2024-06-02,2024-06-01,0
BRA,KER,1c3f15199b3360b3e622fd5ffbe33de1,2024-11-19,2024-06-25,0
BRA,SKI,08170c61ef614f572f38ebe85025629b,2024-06-02,2024-06-01,0
BRA,SKI,3c7ba740726bc6f896f36f36addd1795,2024-06-06,2024-06-07,0
CHI,KER,4feb0887ba19392c126dff5090bafb64,2024-06-04,2024-06-03,0
CHI,KIE,c7fe102ccfcb99bdc0f2ea7cd931952d,2024-07-23,2024-06-04,0
CHI,LAN,1d8cc66d152e298ae5e1b5576af49aff,null,2024-06-06,0
CHI,LAN,35df2c3b1ceead2f4cc36c8bc002a41b,null,2024-06-02,0
CHI,LAN,4c621c64521e9e401a71b6dd1c7dadaf,2024-06-04,2024-06-03,0
CHI,LAN,4c8c4ed34e7b44758194c663301b496e,null,2024-06-01,0


In [0]:
%sql
select 
  status, count(distinct source_customer_id) as counts
from tb1_vw
group by status

status,counts
1,444151
0,7278


In [0]:
%sql
select --*
  brand_country,brand_code,source_name,source_customer_id,date(created_dt) as created_dt,date(last_modified_dt) as last_modified_dt
from prod_latam_catalog.crm_reporting.dim_customer_hist
where source_customer_id in ('fa2acb8f0b053dac8280a02bccada84f')
  -- and brand_country = 'CHI'  
  -- and brand_code = 'LRP'
order by created_dt

brand_country,brand_code,source_name,source_customer_id,created_dt,last_modified_dt
CHI,LRP,SFMC_LANDINGPAGE_LOREAL,fa2acb8f0b053dac8280a02bccada84f,2025-01-05,2025-01-05


In [0]:
%sql
select source_customer_id,brand_country,brand_code,interaction_sub_type,date(interaction_dt) as interaction_dt,source_name
from prod_latam_catalog.crm_reporting.fact_customer_interactions
-- WHERE source_customer_id in ('6eddb4ab90a32d1bd5473ea62bda819a','589496032c6b74b0fe9bf374dd897338') 
where source_customer_id in ('fa2acb8f0b053dac8280a02bccada84f')
  -- and interaction_sub_type = 'Profile Created' 
  and brand_country = 'CHI'  
  and brand_code = 'LRP'

source_customer_id,brand_country,brand_code,interaction_sub_type,interaction_dt,source_name
fa2acb8f0b053dac8280a02bccada84f,CHI,LRP,Profile Created,2024-06-02,SFMC_LANDINGPAGE_LOREAL
fa2acb8f0b053dac8280a02bccada84f,CHI,LRP,Diagnostics,2024-06-02,SFMC_LANDINGPAGE_LOREAL
fa2acb8f0b053dac8280a02bccada84f,CHI,LRP,Profile Updated,2025-01-05,SFMC_LANDINGPAGE_LOREAL
fa2acb8f0b053dac8280a02bccada84f,CHI,LRP,Diagnostics,2025-01-05,SFMC_LANDINGPAGE_LOREAL


In [0]:
%sql
select --*
  brand_country,brand_code,source_name,source_customer_id,date(created_dt) as created_dt,date(last_modified_dt) as last_modified_dt
from prod_latam_catalog.crm_reporting.dim_customer
where source_customer_id in ('fa2acb8f0b053dac8280a02bccada84f')
  -- and brand_country = 'CHI'  
  -- and brand_code = 'LRP'
order by created_dt

brand_country,brand_code,source_name,source_customer_id,created_dt,last_modified_dt
CHI,LRP,SFMC_LANDINGPAGE_LOREAL,fa2acb8f0b053dac8280a02bccada84f,2025-01-05,2025-01-05
